# EMBER-Dyn Quickstart 03 — 3-layer MLP (PyTorch, template)

**Dataset**: EMBER-Dyn (T1 binary detection)  
**Expected metric**: test ROC AUC in the ~0.97--0.99 range; exact
value depends on architecture and optimisation choices.  
**Runtime**: ~3 minutes on Colab free CPU tier; faster on GPU.

Template for users who want a neural-net baseline. The paper
Table 4 reports LightGBM only; use this notebook to produce an
MLP row for your own comparison.

In [ ]:
!pip install -q pandas pyarrow scikit-learn torch

In [ ]:
import pandas as pd, numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# EMBER-Dyn open tier is archived on Zenodo (DOI in the README). Download it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
DATA_DIR = 'dataset/static'
df_train = pq.read_table(f'{DATA_DIR}/train.parquet').to_pandas()
df_val   = pq.read_table(f'{DATA_DIR}/val.parquet').to_pandas()
df_test  = pq.read_table(f'{DATA_DIR}/test.parquet').to_pandas()

feat_cols = [c for c in df_train.columns if c.startswith('feature_')]
def split_X_y(df):
    return df[feat_cols].values.astype('float32'), df['label'].values.astype('int64')
X_train, y_train = split_X_y(df_train)
X_val,   y_val   = split_X_y(df_val)
X_test,  y_test  = split_X_y(df_test)
print(f'train {X_train.shape}  val {X_val.shape}  test {X_test.shape}')

## Define and train a 3-layer MLP

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

class MLP(nn.Module):
    def __init__(self, d_in, d_hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(d_hidden, d_hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(d_hidden // 2, 2),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(d_in=X_train.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=512, shuffle=True,
)

for epoch in range(15):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
    if (epoch + 1) % 3 == 0:
        print(f'epoch {epoch+1:2d}  train loss {loss.item():.4f}')

## Evaluate on the test split

In [ ]:
model.eval()
with torch.no_grad():
    p_test = torch.softmax(model(torch.from_numpy(X_test).to(device)), dim=1)[:, 1].cpu().numpy()
print(f'MLP-3  test ROC AUC = {roc_auc_score(y_test, p_test):.4f}    '
      f'PR AUC = {average_precision_score(y_test, p_test):.4f}    '
      f'F1 = {f1_score(y_test, p_test > 0.5):.4f}')